In [34]:
from dotenv import load_dotenv
import os

load_dotenv()

WEATHER_API_KEY=os.getenv("WEATHER_API_KEY")

In [35]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
import requests

In [36]:
search_tool=DuckDuckGoSearchRun()

search_tool.invoke("what is the name  of the education minister of india")


"13 May 2026 · Meet Dharmendra Pradhan: the education minister of India. India is filled with rogue politicians and business. its just about making money, having a lavish ... 25 Jul 2026 · Pralhad Joshi has been appointed as India's new Union Education Minister, taking charge of the ministry following Dharmendra Pradhan's resignation. Along with ... 13 May 2026 · Dharmendra Pradhan Appointed as Education Minister In Wednesday's cabinet reshuffle, 'Ujjawala Man' Dharmendra Pradhan took over the Education Ministry and ... 4 Aug 2026 · The current education minister of India in 2026 is Dharmendra Pradhan. He has been serving in this role since July 2021 25 Jul 2026 · Union Minister Pralhad Joshi has been given additional charge of the Education Ministry following the resignation of Dharmendra Pradhan."

In [37]:
@tool
def get_weather_data(city:str)->str:
    """
    THIS FUNCTION FETCHES THE CURRENT WEAHER DATA FOR A GIVEN CITY
    """
    url=f"http://api.weatherstack.com/current?access_key=WEATHER_API_KEY&query={city}"

    response=requests.get(url)

    return response.json()


In [38]:
llm=ChatGroq(model="openai/gpt-oss-120b")

In [39]:
from langchain.agents import create_agent

In [45]:
#this part is generated by AI

system_prompt = """
You are a helpful and intelligent AI assistant with access to two tools:

1. weather_tool:
   - Use this tool when the user asks about weather, temperature,
     rainfall, humidity, wind, forecast, or other weather-related information.
   - Always use the weather tool when current or forecast weather data is required.
   - Do not guess or invent weather information.

2. search_tool:
   - Use this tool when the user asks for current, recent, or web-based information.
   - Use it to search DuckDuckGo for information that you do not reliably know.
   - Use it for news, current events, people, organizations, facts, or any information
     that may have changed recently.
   - Do not invent information when a web search can provide the answer.

General instructions:
- Understand the user's question before deciding which tool to use.
- Use the appropriate tool when necessary.
- If both weather and web information are required, you may use both tools.
- After receiving tool results, analyze them and provide a clear, concise answer.
- Do not expose internal tool calls or implementation details to the user.
- If the tools do not provide enough information, clearly state that.
- For simple questions that you already know reliably, answer directly without using a tool.
- Prioritize accurate and up-to-date information over guessing.
"""

In [41]:
agent=create_agent(
    model=llm,
    tools=[search_tool,get_weather_data],
    system_prompt=system_prompt
)

In [42]:
response=agent.invoke({"messages":[
    {"role":"user","content":"what is the capital of odisha and whst is the temperature now"}
]})

In [43]:
response["messages"][-1]

AIMessage(content='The capital of Odisha is **Bhubaneswar**.  \n\nAccording to the most recent weather information available from online sources, the temperature in Bhubaneswar right now is roughly **24\u202f°C to 26\u202f°C** (about 75\u202f°F). Different live feeds report values in that range, so you can expect the temperature to be around the mid‑20s Celsius at the moment.', additional_kwargs={'reasoning_content': 'We need answer: capital of Odisha is Bhubaneswar. Temperature now: from search results we have multiple values: one says 26.1°C, another says 24°C. Need most recent. The second search shows "current weather in Bhubaneswar is 24°C". The first shows 26.1°C. Could be different times. We should perhaps mention approximate current temperature around 25°C, and note variability. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 183, 'prompt_tokens': 1146, 'total_tokens': 1329, 'completion_time': 0.379208233, 'completion_tokens_details': {'reasoning_token